# **Last week's homework:**

Use the model definition below and train the model **without using an optimizer**.
Instead, use the *.grad* property of the tensors and perform the parameter update yourself.



In [52]:
import torch
import torch.nn as nn

###Simple Linear Model (2 parameters, 1 input, 1 output)
#compare __init__() from: https://pytorch.org/docs/stable/_modules/torch/nn/modules/linear.html#Linear
class linFunc(nn.Module):
    def __init__(self):
        super().__init__()
        #parameter W
        self.paramW = nn.Parameter(torch.zeros(1,requires_grad=True))
        #parameter b
        self.paramB = nn.Parameter(torch.zeros(1,requires_grad=True))

    def forward(self, x):
        #function: (W * x) + b | where x is the input, W and b the parameters to be learned
        return self.paramW * x + self.paramB

#Training loop (not fully implemented):

samples = torch.FloatTensor([[[2],[0]],[[3],[1]],[[4],[2]],[[5],[3]]])
model = linFunc()
lossFct = nn.MSELoss()
learningRate = 0.01

ins =  samples[:,0] #take the inputs from samples
tgts = samples[:,1] #take the targets from samples

for x in range(2500): #we train for 2500 epochs
  preds = model(ins) #prediction
  loss = lossFct(preds,tgts)
  loss.backward()
  model.paramW = nn.Parameter(model.paramW - (model.paramW.grad * learningRate))
  model.paramB = nn.Parameter(model.paramB - (model.paramB.grad * learningRate))

#searched: W = 1 and b = -2
print (model.paramB)
print (model.paramW)

Parameter containing:
tensor([-1.9724], requires_grad=True)
Parameter containing:
tensor([0.9928], requires_grad=True)


Performing parameter updates in parallel:

In [53]:
import torch
import torch.nn as nn

###Simple Linear Model (2 parameters, 1 input, 1 output)
#compare __init__() from: https://pytorch.org/docs/stable/_modules/torch/nn/modules/linear.html#Linear
class linFunc(nn.Module):
    def __init__(self):
        super().__init__()
        #parameters
        self.params = nn.Parameter(torch.zeros(2,requires_grad=True)) #put all parameters together

    def forward(self, x):
        #function: (W * x) + b | where x is the input, W and b the parameters to be learned
        return self.params[0] * x + self.params[1]

#Training loop (not fully implemented):

samples = torch.FloatTensor([[[2],[0]],[[3],[1]],[[4],[2]],[[5],[3]]])
model = linFunc()
lossFct = nn.MSELoss()
learningRate = 0.01

ins =  samples[:,0] #take the inputs from samples
tgts = samples[:,1] #take the targets from samples

for x in range(2500): #we train for 2500 epochs
  preds = model(ins) #prediction
  loss = lossFct(preds,tgts)
  loss.backward()
  model.params = nn.Parameter(model.params - (model.params.grad * learningRate)) #update the parameters in parallel!


#searched: W = 1 and b = -2
print (model.params[0])
print (model.params[1])

tensor(0.9928, grad_fn=<SelectBackward0>)
tensor(-1.9724, grad_fn=<SelectBackward0>)


# **Training, Validation, Testing**
- https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_20newsgroups.html#sklearn.datasets.fetch_20newsgroups
- https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction
- https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html?highlight=vectorizer#sklearn.feature_extraction.text.CountVectorizer

Ideally, data is split into three subsets: Train, Validation and Test

- Training: The part of the data to perform the training (parameter updates)
- Validation: Part of the data to verify that training generalizes (during training)
- Test: Part of the data to check model performance (after training)

We will perform a text classification task on the 20Newsgroups dataset. Each text from the dataset belongs to a certain category. These categories will function as the supervision signal and be the classification targets.

The total number of categories is 20:

- 'alt.atheism'
- 'comp.graphics'
- 'comp.os.ms-windows.misc'
- 'comp.sys.ibm.pc.hardware'
- 'comp.sys.mac.hardware'
- 'comp.windows.x'
- 'misc.forsale'
- 'rec.autos'
- 'rec.motorcycles'
- 'rec.sport.baseball'
- 'rec.sport.hockey'
- 'sci.crypt'
- 'sci.electronics'
- 'sci.med'
- 'sci.space'
- 'soc.religion.christian'
- 'talk.politics.guns'
- 'talk.politics.mideast'
- 'talk.politics.misc'
- 'talk.religion.misc'

In [27]:
from sklearn.datasets import fetch_20newsgroups
categories = ['comp.graphics', 'sci.med']
twenty_test = fetch_20newsgroups(subset='test', categories=categories, shuffle=True, random_state=42)
print (twenty_test.data[5])
print (twenty_test.target[5])
print (twenty_test.target_names)

From: noring@netcom.com (Jon Noring)
Subject: Adenocarcinoma of the Lungs
Organization: Netcom Online Communications Services (408-241-9760 login: guest)
Lines: 34

Putting aside our substantial differences, I'd like to ask the knowledgeable
ones to give feedback on this.  Let me explain.

One of my family members last week was discovered to have a brain tumor after
having some difficulties with walking and writing (she is 64 years old).
Otherwise, she is in fine health.  The discovery was made via CAT scans.

She then had MRI scans done, where small cancerous areas were discovered
in her lungs.  Biopsies showed it to be adenocarcinoma.  One spot is
in the lungs, and another in the pneumothorax.  The oncologists believe
the cancer started in the lungs and caused the brain tumor (she smoked
until four years ago).

Anyway, I'd like feedback as to what adenocarcinoma is, how it is different
from other cancers, how she will be treated (luckily the tumor is right
below the skull and can be 

Model definition (the details will be the subject of the upcoming lecture)

In [62]:
class linearClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        #linear Layer
        self.lin = nn.Linear(32,2) #note that the input dimension must match the feature dimension of the data!

    def forward(self, x):
        return self.lin(x)

Data preparation:

In [63]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

def loadAndTransform(tfidf = True,featureIns = 32):
    #restrict to two classes
    categories = ['comp.graphics', 'sci.med']
    #load data
    twenty_train = fetch_20newsgroups(subset='train', categories=categories, shuffle=True, random_state=42)
    twenty_test = fetch_20newsgroups(subset='test', categories=categories, shuffle=True, random_state=42)
    #limit features to 32
    if tfidf:
        vect = TfidfVectorizer(max_features=featureIns)
    else:
        vect = CountVectorizer(max_features=featureIns)
    # vectorize training data
    X_train = vect.fit_transform(twenty_train.data).toarray() #note .fit_transform vs. .transform!!!
    # vectorize test data
    X_test = vect.transform(twenty_test.data).toarray()
    # select the targets for both training and test
    y_test = twenty_test.target
    y_train = twenty_train.target
    return X_train,X_test,y_test,y_train

Loading the data

In [64]:
from sklearn.metrics import confusion_matrix, f1_score
X_train,X_test,y_test,y_train = loadAndTransform(False) #Call the function for data

trainIns = torch.FloatTensor(X_train)
testIns = torch.FloatTensor(X_test)
trainTgt = torch.LongTensor(y_train)
testTgt = torch.LongTensor(y_test)

Note the different shapes for Input and Target Tensors!

In [65]:
print (testTgt.shape)
print (testIns.shape)

torch.Size([785])
torch.Size([785, 32])


Creating a Validation set for showcasing purposes:

In [66]:
evalTgt = testTgt[:400]
evalIns = testIns[:400]

testTgt = testTgt[400:]
testIns = testIns[400:]

Train Loop:

In [70]:
model = linearClassifier()
lossFct = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(),lr=0.01)
###TRAINING
for x in range(500): #250 epochs training
    preds = model(trainIns)
    loss = lossFct(preds,trainTgt)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    ###EVALUATION
    if x % 50 == 0:
      with torch.no_grad():
        print (f"Training Loss in Epoch {x}:\n{float(loss)}")
        predVal = model(evalIns)
        evalLoss = lossFct(predVal,evalTgt)
        print (f"Evaluation Loss in Epoch {x}:\n{float(evalLoss)}")

###TESTING
with torch.no_grad(): #deactivate gradients for evaluation
    testPreds = model(testIns) #predict
    testPreds = torch.topk(testPreds,1).indices.view(-1) #transform predictions to compatible format
    print ("\nResults - PyTorch - Linear Model")
    print (confusion_matrix(testTgt,testPreds))
    print (f1_score(testTgt,testPreds,average='macro'))

Training Loss in Epoch 0:
0.8952229619026184
Evaluation Loss in Epoch 0:
0.9598713517189026
Training Loss in Epoch 50:
0.6206944584846497
Evaluation Loss in Epoch 50:
0.6994961500167847
Training Loss in Epoch 100:
0.5859145522117615
Evaluation Loss in Epoch 100:
0.6954541206359863
Training Loss in Epoch 150:
0.5696021914482117
Evaluation Loss in Epoch 150:
0.6928942203521729
Training Loss in Epoch 200:
0.5642611384391785
Evaluation Loss in Epoch 200:
0.6994286179542542
Training Loss in Epoch 250:
0.5591745972633362
Evaluation Loss in Epoch 250:
0.6996617317199707
Training Loss in Epoch 300:
0.5559728741645813
Evaluation Loss in Epoch 300:
0.7009713649749756
Training Loss in Epoch 350:
0.5538159012794495
Evaluation Loss in Epoch 350:
0.7025629281997681
Training Loss in Epoch 400:
0.5523499846458435
Evaluation Loss in Epoch 400:
0.7041552662849426
Training Loss in Epoch 450:
0.5513910055160522
Evaluation Loss in Epoch 450:
0.7055726647377014

Results - PyTorch - Linear Model
[[136  41]
 

Model output:

In [78]:
preds = model(trainIns[0])
print (preds) #raw predictions
smax = nn.Softmax(dim=0)
print (smax(preds)) #softmax -> probability distribution

print (trainTgt[0]) #target

tensor([0.7082, 0.9121], grad_fn=<ViewBackward0>)
tensor([0.4492, 0.5508], grad_fn=<SoftmaxBackward0>)
tensor(0)


**Homework:**

See how the following measures (or a combination of them) affect the model performance:

- Changing the learning rate
- Changing the number of features (note: change both in the model definition and in the loadAndTransform() function!)
- Try out different categories from the data set
- Try out different vectorizers